# MLOps architecture on Databricks

This notebook is the map for the blueprint: the end-to-end MLOps loop on Databricks in one place,
and the two architectural decisions that wrap it. It carries no code. The worked example that
implements this loop runs across the notebooks that follow, each building one stage of it for real
on Databricks. The diagram below tags every stage with the notebook that does so. The exception is
`06_agent_patterns`, a separate track for retrieval-augmented agents that does not fit the churn
loop.

I read this first to see the whole system, then follow the worked example from `02` onward. Each
later notebook opens by pointing back to this diagram and stating which stages it covers.

**Prerequisites.** `00_dev_workflows` (how code runs on Databricks).

**What it covers.**
1. The end-to-end reference architecture: raw data -> features -> train -> registry -> serve -> monitor.
2. Deploy-code vs deploy-model: what crosses the environment boundary.
3. The dev / staging / prod strategy that wraps the loop.

## The reference architecture

A single customer record moves through the loop: a raw row becomes features, trains a model, which
is registered to Unity Catalog, served to score new customers, and monitored for drift. When the
data shifts, monitoring triggers retraining and the loop repeats. Unity Catalog governs every stage
as the single boundary for data, features, models, and lineage.

```mermaid
graph LR
  RAW[Raw churn] --> FEAT[Features 03]
  FEAT --> TRAIN[Train + MLflow 03]
  TRAIN --> REG[UC registry 03]
  REG --> SERVE[Serve 04]
  SERVE --> MON[Monitoring 04]
  MON -->|drift triggers retrain| TRAIN
  UC[Unity Catalog governance] -.-> FEAT
  UC -.-> REG
  UC -.-> SERVE
```

The number on each stage is the notebook that builds it for real. The training pipeline (`03`)
covers features, training, and registration; serving and monitoring (`04`) cover scoring, drift, and
the retrain trigger. Config (`02`) parameterizes the pipeline, and the deploy capstone (`05`) wraps
the loop in dev/staging/prod targets and CI/CD.

## Deploy-code vs deploy-model

The first architectural decision is what crosses the environment boundary from dev to prod.

- **Deploy-code (the blueprint default).** I promote the code. The training pipeline runs in each
  environment, so the model is retrained and registered in prod from prod data.
- **Deploy-model.** I promote the model artifact. Training runs once, usually in dev, and the
  resulting model is promoted through staging to prod.

```mermaid
graph TD
  CC[deploy-code: promote CODE] --> C1[dev: train]
  C1 --> C2[staging: train + test]
  C2 --> C3[prod: train + register]
  MM[deploy-model: promote ARTIFACT] --> M1[dev: train + register]
  M1 --> M2[staging: validate]
  M2 --> M3[prod: deploy artifact]
```

| Criterion | Favors deploy-code | Favors deploy-model |
|---|---|---|
| What is promoted / where training runs | Code; model trained in each env (incl. prod) | A model artifact; trained once, usually in dev |
| Retraining cadence | Frequent or automated retrains | Rare; expensive or manual training |
| Training-pipeline parity | Exercised in staging exactly as in prod | Training path not re-run downstream |
| Regulatory reproducibility | Pipeline reproducible from versioned code | Exact artifact pinned and promoted as-is |
| Training cost | Acceptable to retrain per environment | Very high; train once, reuse |
| Production data access | Prod data reachable for in-env training | Prod data restricted; train where data lives |

I default to deploy-code. It preserves train/serve parity and exercises the whole pipeline in
staging as prod runs it. I reach for deploy-model when training is too costly to repeat, or when
prod data cannot be reached from the training step. The deploy capstone (`05`) operationalizes this
posture with Asset Bundle targets.

## dev / staging / prod

The loop runs in three environments, separated by Unity Catalog catalog and by bundle target (the
deploy capstone, `05`). The code is identical across environments; only the catalog, and so the data
and models it sees, changes. Aliases are scoped to their catalog, so `@champion` in prod is
independent of `@champion` in dev.

| Environment | Catalog | Workspace | Data | Who promotes |
|---|---|---|---|---|
| dev | `dev` (or a sandbox) | shared dev | sampled / synthetic | any engineer |
| staging | `staging` | staging | prod-like, governed | CI on PR merge |
| prod | `prod` | prod (often isolated) | full production | gated CD / approver |

The worked example uses `main.default` so it runs in any workspace with zero setup. In a real
deployment the catalog is swapped per environment, wired by the bundle target.

## Further reading

[`docs/links.md`](docs/links.md) holds the full annotated list. For this notebook:

- [Big Book of MLOps](https://www.databricks.com/resources/ebook/the-big-book-of-mlops): the source of the deploy-code vs deploy-model framing.
- [MLOps on Databricks (recommended architecture)](https://docs.databricks.com/machine-learning/mlops/mlops-workflow.html): the canonical reference workflow.

The worked example that implements this loop: `02` config, `03` the training pipeline (features,
MLflow, registry), `04` serving and monitoring, and `05` deploy and promotion with Asset Bundles.